# 🧬 ⚙️ CpGPT Data Denoising Tutorial ⚙️ 🧬

Welcome to the CpGPT Data Denoising Tutorial! 👋 

In this notebook, we'll walk you through the process of denoising the methylome from the typically noisy arrays with a pre-trained CpGPT model.

## Table of Contents

1. [Setup Environment](#1-setup-environment)
2. [Load CpGPT Dependencies](#2-load-cpgpt-dependencies)
3. [Load Pre-trained Model](#3-load-pre-trained-model)
4. [Save Data of Interest](#4-save-data-of-interest)
5. [Create Dataset](#5-create-dataset)
6. [Convert Illumina Probes](#6-convert-illumina-probes)
7. [Generate DNA LLM Embeddings](#7-generate-dna-llm-embeddings)
8. [Embed and Denoise Methylome](#8-embed-and-denoise-methylome)
9. [Visualize Denoised Methylome](#9-visualize-denoised-methylome)
10. [Next Steps](#10-next-steps)

## 1. Setup Environment

We'll import the necessary Python packages and set up our environment for CpGPT. We'll be using a mix of standard data science libraries and CpGPT-specific modules. We'll also set some important variables that will be used throughout the notebook. Pay attention to these as you may need to adjust them based on your specific setup and requirements. The bulk of the variables are defined here below. The pre-trained model checkpoint alongside DNA embedding files can also be downloaded if not present.

CpGPT model files and DNA-sequence dependencies are hosted on Hugging Face.
The download cell below downloads any missing files into the standard Hugging Face cache and reuses them on later runs; no command-line download is needed.

### 1.1 Download dependencies and define variables

In [ ]:
from cpgpt import download_cpgpt

resources = download_cpgpt(model="small", species="human")

In [ ]:
# Random seed for reproducibility
RANDOM_SEED = 42

# Directory paths
PROCESSED_DIR = "../data/tutorials/processed/denoise"
ARROW_DF_PATH = "../data/GSE53051/raw/betas/gse_betas.arrow"
ARROW_METADATA_PATH = "../data/GSE53051/raw/metadata/metadata.arrow"

# The maximum context length to give to the model
MAX_INPUT_LENGTH = 10_000

# Device configuration
DEVICE = "cuda"

LLM_DEPENDENCIES_DIR = str(resources.dependencies_path)
DEPENDENCIES_DIR = str(resources.dependencies_path)
MODEL_CHECKPOINT_PATH = str(resources.checkpoint_path)
MODEL_CONFIG_PATH = str(resources.config_path)
MODEL_VOCAB_PATH = str(resources.vocab_path) if resources.vocab_path is not None else None

> **⚠️ Warning**
> 
> It is recommended to have a GPU for inference as CPU might be slow.
> 
> Reconstructing the methylome for a few hundred samples might take up to one hour on a CPU. ⌛
>
> This might be a great exercise in testing your patience.

### 1.2 Import packages


In [ ]:
# Standard library imports
import os
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

# Plotting imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use(
    [
        "science",
        "nature",
    ],
)
import pyaging as pya
import seaborn as sns

# Lightning imports
from lightning.pytorch import seed_everything
from scipy import stats
from scipy.stats import pearsonr, spearmanr

# cpgpt-specific imports
from cpgpt.data.components.cpgpt_datasaver import CpGPTDataSaver
from cpgpt.data.components.cpgpt_dataset import CpGPTDataset
from cpgpt.data.components.dna_llm_embedder import DNALLMEmbedder
from cpgpt.data.components.illumina_methylation_prober import IlluminaMethylationProber
from cpgpt.infer.cpgpt_inferencer import CpGPTInferencer
from cpgpt.utils.rich_utils import create_rich_dataset_preview

# Set random seed for reproducibility
seed_everything(RANDOM_SEED, workers=True)

## 2. Load CpGPT Dependencies

Now that we have our basic environment set up, we need to load three crucial components for CpGPT:

1. **DNA LLM Embedder**: This component contains the embeddings for all CpGs of interest. It's essential for converting DNA sequences into a format that our model can understand and process.

2. **Illumina Methylation Prober**: This tool allows us to convert probe names to genomic locations, which is required for working with Illumina methylation array data.

3. **CpG Inferencer**: This component is responsible for loading the pre-trained CpGPT model checkpoint and any additional configurations or overrides needed for the fine-tuning process.

### 3.1 Load DNALLMEmbedder

In [ ]:
embedder = DNALLMEmbedder(dependencies_dir=DEPENDENCIES_DIR)

### 3.2 Load IlluminaMethylationProber

In [ ]:
prober = IlluminaMethylationProber(dependencies_dir=DEPENDENCIES_DIR, embedder=embedder)

### 3.3 Load CpGInferencer

In [ ]:
inferencer = CpGPTInferencer()

## 3. Load Pre-trained Model

We need to load a pre-trained CpGPT model to be able to reconstruct the beta values for different CpG sites. Generally, the pre-trained model works well for zero-shot denoising but you might want to finetune the model using data that is more appropriate for your denoising goals.

In [ ]:
config = inferencer.load_cpgpt_config(MODEL_CONFIG_PATH)
model_info = {"current_hparams": config}
model = inferencer.load_cpgpt_model(
    config,
    model_ckpt_path=MODEL_CHECKPOINT_PATH,
    strict_load=True,
)

## 4. Save Data of Interest

We'll prepare our methylation data for denoising and save it in a format that CpGPT can efficiently process. We first begin with a dataset that is already stored in .arrow (feather v2) format. Then, let's create a memory-mapped dataset.

In [ ]:
# Read the arrow file
df = pd.read_feather(ARROW_DF_PATH)
metadata_df = pd.read_feather(ARROW_METADATA_PATH)

# Only keep samples with age information
metadata_df = metadata_df[metadata_df["age"] != "na"]
df = df.loc[metadata_df.index]

# Save age and female columns
age = metadata_df["age"]
female = metadata_df["Sex"] != "m"

# Filter df to only include CpGs from the Illumina Prober
df = df.loc[
    :,
    np.intersect1d(df.columns, list(prober.illumina_metadata_dict["homo_sapiens"].keys())),
]

# Let's make sure that we are only use CpG sites that are seen by the model
df = df.iloc[:, :MAX_INPUT_LENGTH]

# Save df to arrow file
Path(PROCESSED_DIR).mkdir(parents=True, exist_ok=True)
SUBSET_DF_PATH = os.path.join(PROCESSED_DIR, "df_subset.arrow")
df.to_feather(SUBSET_DF_PATH)

# Show first few rows
df.head()

In [ ]:
# Define datasaver
denoise_datasaver = CpGPTDataSaver(data_paths=SUBSET_DF_PATH, processed_dir=PROCESSED_DIR)

# Process the file
denoise_datasaver.process_files(prober, embedder)

## 5. Generate DNA LLM Embeddings
 
In this section, we'll generate any DNA embeddings for the genomic locations of our dataset that might be missing. The model requires that all genomic locations have embeddings from a DNA LLM or it will fail otherwise. The easiest way is to download such embeddings if you are working with human samples ([step 1.3](#1.3-download-dependencies)) and Illumina arrays or it might take a little bit for the processing to happen.

In [ ]:
# Get all unique species from the datasavers
all_species = list(denoise_datasaver.all_genomic_locations.keys())

# Loop through each species
for species in all_species:
    all_genomic_locations = list(denoise_datasaver.all_genomic_locations.get(species, []))

    embedder.parse_dna_embeddings(
        all_genomic_locations,
        species,
        dna_llm=model_info["current_hparams"]["data"]["dna_llm"],
        dna_context_len=model_info["current_hparams"]["data"]["dna_context_len"],
    )

## 6. Create Dataset

We need to define a way that the CpGPT model can efficiently access our memory-mapped dataset. Let's define a CpGPTDataset object and check the output.

In [ ]:
denoise_data = CpGPTDataset(
    embedder,
    processed_dir=PROCESSED_DIR,
    max_length=MAX_INPUT_LENGTH,
    sorting_strategy=model_info["current_hparams"]["data"]["sorting_strategy"],
    dna_context_len=model_info["current_hparams"]["data"]["dna_context_len"],
    dna_llm=model_info["current_hparams"]["data"]["dna_llm"],
)

Let's double check the outputs make sense:

In [ ]:
create_rich_dataset_preview(denoise_data, "Denoise Data Sample")

## 7. Convert Illumina Probes

In order for the model to understand what part of the methylome needs to be reconstructed, a genomic location is necessary. This can be given directly in the format '12:213103' or first converted from the probes of the Illumina methylation arrays. The IlluminaMethylationProber class has a handy function for that conversion for the most up-to-date arrays as of the publication of this tutorial. As an example, let's get the Illumina probes for some epigenetic clocks using pyaging.

In [ ]:
# Get all probes
probes = df.columns

# Convert valid probes to genomic locations
genomic_locations = prober.locate_probes(probes, "homo_sapiens")

# Show first few locations
genomic_locations[0:5]

## 8. Embed and Denoise Methylome

First, let's use CpGPT to create a sample embedding given the available information in the CpG sites. Then, let's ask it to reconstruct the beta values for the genomic locations of interest.

In [ ]:
# Embed samples
sample_embeddings = inferencer.embed_sample(model, denoise_data, batch_size=1)

In [ ]:
# Reconstruct denoised beta values
X_denoised, X_unc = inferencer.reconstruct_betas(
    model,
    sample_embedding=sample_embeddings,
    genomic_locations=genomic_locations,
    embedder=embedder,
    dna_llm=model_info["current_hparams"]["data"]["dna_llm"],
    dna_context_len=model_info["current_hparams"]["data"]["dna_context_len"],
    species="homo_sapiens",  # You need to pick a single species
    batch_size=1,
)

In [ ]:
# Transform to DataFrame
X_denoised_df = pd.DataFrame(X_denoised.detach().cpu().numpy(), columns=probes, index=df.index)
X_unc_df = pd.DataFrame(X_unc.detach().cpu().numpy(), columns=probes, index=df.index)

## 9. Visualize Denoised Methylome

As a quick sanity check, let's create some plots to try to understand what is going on. We'll visualize:

1. The correlation between the denoised and original values
2. The relationship between the uncertainty and the difference between the denoised and original values
3. The predictions of epigenetic clocks from the denoised values

These visualizations will help us ensure that our model is working as expected. If not, it might be worth finetuning with the appropriate data.

### 9.1 Denoised vs original values

In [ ]:
# Create subplots
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Original vs Denoised Beta Values")

# Colors for each sample
colors = ["blue", "red", "green"]

# Plot for each of the first 3 samples
for i in range(3):
    x = df.iloc[i]
    y = X_denoised_df.iloc[i]

    # Calculate Pearson and Spearman correlations
    pearson_corr, _ = stats.pearsonr(x, y)
    spearman_corr, _ = stats.spearmanr(x, y)

    # Scatter plot
    axs[i].scatter(x, y, c=colors[i], s=2, alpha=0.5)

    # Trend line
    z = np.polyfit(x, y, 1)
    p = np.poly1d(z)
    axs[i].plot(x, p(x), color="black", linewidth=2)

    # Add correlation text
    axs[i].text(
        0.05,
        0.95,
        f"Pearson: {pearson_corr:.3f}\nSpearman: {spearman_corr:.3f}",
        transform=axs[i].transAxes,
        verticalalignment="top",
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.8, "edgecolor": "black"},
    )

    axs[i].set_title(f"Sample {i+1}")
    axs[i].set_xlabel("Original Beta Values")
    axs[i].set_ylabel("Denoised Beta Values")

plt.tight_layout()
plt.show()

In [ ]:
# Calculate performance metrics between original and denoised values


# Function to calculate metrics for a single sample
def calculate_metrics(original, denoised):
    pearson_corr, _ = stats.pearsonr(original, denoised)
    spearman_corr, _ = stats.spearmanr(original, denoised)
    mae = np.mean(np.abs(original - denoised))
    return pearson_corr, spearman_corr, mae


# Calculate metrics per sample
sample_metrics = []
for i in range(len(df)):
    original = df.iloc[i]
    denoised = X_denoised_df.iloc[i]
    metrics = calculate_metrics(original, denoised)
    sample_metrics.append(metrics)

sample_metrics_df = pd.DataFrame(sample_metrics, columns=["Pearson", "Spearman", "MAE"])

# Set up the matplotlib figure
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Performance Metrics Across Samples", fontsize=16)

# Colors for each metric
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]  # Blue, Orange, Green

# Create violin plots for each metric
for i, metric in enumerate(["Pearson", "Spearman", "MAE"]):
    sns.violinplot(y=sample_metrics_df[metric], ax=axes[i], color=colors[i], inner="box")
    sns.stripplot(
        y=sample_metrics_df[metric],
        ax=axes[i],
        color="black",
        size=3,
        alpha=0.5,
        jitter=0.3,
    )

    axes[i].set_title(f"{metric} Correlation" if metric != "MAE" else "Mean Absolute Error")
    axes[i].set_xlabel("Sample Distribution")
    axes[i].set_ylabel("")  # Remove y-axis label

    # Set background color using a valid color format
    axes[i].set_facecolor((0.94, 0.94, 0.94, 0.95))  # Light gray with 95% opacity

    # Add grid
    axes[i].grid(True, linestyle="--", alpha=0.7)

    # Calculate statistics
    mean_val = sample_metrics_df[metric].mean()
    median_val = sample_metrics_df[metric].median()
    max_val = sample_metrics_df[metric].max()
    min_val = sample_metrics_df[metric].min()

    # Add statistics to the plot
    stats_text = (
        f"Mean: {mean_val:.3f}\nMedian: {median_val:.3f}\nMax: {max_val:.3f}\nMin: {min_val:.3f}"
    )
    axes[i].text(
        0.05,
        0.95,
        stats_text,
        transform=axes[i].transAxes,
        verticalalignment="top",
        bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.8},
    )

# Adjust layout
plt.tight_layout()
plt.show()

### 9.2 Uncertainty vs removed noise

In [ ]:
# Calculate the difference between original and denoised values for the first three samples
diff_df = df.iloc[:3] - X_denoised_df.iloc[:3]

# Create subplots for the first three samples
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    "Uncertainty vs Difference between Original and Denoised Values (First 3 Samples)",
    fontsize=16,
)

# Define colors for each sample
colors = ["red", "green", "blue"]

for i in range(3):
    # Flatten the dataframes to 1D arrays for each sample
    uncertainty_values = X_unc_df.iloc[i].values
    diff_values = diff_df.iloc[i].values

    # Ensure both arrays have the same shape
    min_length = min(len(uncertainty_values), len(diff_values))
    uncertainty_values = uncertainty_values[:min_length]
    diff_values = diff_values[:min_length]

    # Remove NaN values
    mask = ~np.isnan(uncertainty_values) & ~np.isnan(diff_values)
    uncertainty_values = uncertainty_values[mask]
    diff_values = diff_values[mask]

    # Check if there are at least 2 valid data points
    if len(uncertainty_values) < 2 or len(diff_values) < 2:
        pearson_corr, spearman_corr = np.nan, np.nan
    else:
        # Calculate correlations
        pearson_corr, _ = stats.pearsonr(uncertainty_values, diff_values)
        spearman_corr, _ = stats.spearmanr(uncertainty_values, diff_values)

    # Add scatter plot for each sample
    axes[i].scatter(
        uncertainty_values,
        diff_values,
        c=colors[i],
        alpha=0.5,
        s=20,
        label=f"Sample {i+1}",
    )

    # Add line of best fit
    if len(uncertainty_values) >= 2:
        z = np.polyfit(uncertainty_values, diff_values, 1)
        p = np.poly1d(z)
        x_range = np.linspace(np.min(uncertainty_values), np.max(uncertainty_values), 100)
        axes[i].plot(x_range, p(x_range), color="black", linestyle="--", label="Best Fit")

    # Add correlation annotations
    if not np.isnan(pearson_corr):
        axes[i].text(
            0.05,
            0.95,
            f"Pearson: {pearson_corr:.4f}\nSpearman: {spearman_corr:.4f}",
            transform=axes[i].transAxes,
            verticalalignment="top",
            bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.8, "edgecolor": "gray"},
        )
    else:
        axes[i].text(
            0.05,
            0.95,
            "Insufficient data",
            transform=axes[i].transAxes,
            verticalalignment="top",
            bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.8, "edgecolor": "gray"},
        )

    # Set titles and labels
    axes[i].set_title(f"Sample {i+1}")
    axes[i].set_xlabel("Uncertainty")
    axes[i].set_ylabel("Difference (Original - Denoised)" if i == 0 else "")
    axes[i].legend()

    # Set background color and grid
    axes[i].set_facecolor((0.94, 0.94, 0.94, 0.95))  # Light gray with 95% opacity
    axes[i].grid(True, linestyle="--", alpha=0.7)

# Adjust layout
plt.tight_layout()
plt.show()

### 9.3 Epigenetic clock prediction

In [ ]:
# Define clocks of interest from pyaging
clocks = ["horvath2013", "dnamphenoage", "dunedinpace", "grimage2", "epitoc1", "hannum"]

# Add age and female columns to the dataframes
df["age"] = age
df["female"] = female
X_denoised_df["age"] = age
X_denoised_df["female"] = female

# Ensure all values are numeric
df = df.astype(float)
X_denoised_df = X_denoised_df.astype(float)

# Predict age using original beta values
adata_original = pya.pp.df_to_adata(df, imputer_strategy="mean", verbose=False)
pya.pred.predict_age(adata_original, clocks, verbose=False)

# Predict age using denoised beta values
adata_denoised = pya.pp.df_to_adata(X_denoised_df, imputer_strategy="mean", verbose=False)
pya.pred.predict_age(adata_denoised, clocks, verbose=False)

# Calculate correlations and MAE for each clock
results = {}
fig, axes = plt.subplots(3, 2, figsize=(10, 12))
fig.suptitle("Original vs Denoised Epigenetic Clock Predictions", fontsize=16)

for i, clock in enumerate(clocks):
    if clock in adata_original.obs.columns and clock in adata_denoised.obs.columns:
        original = np.array(adata_original.obs[clock])
        denoised = np.array(adata_denoised.obs[clock])

        pearson_corr, _ = pearsonr(original, denoised)
        spearman_corr, _ = spearmanr(original, denoised)
        mae = np.mean(np.abs(original - denoised))

        results[clock] = {"Pearson": pearson_corr, "Spearman": spearman_corr, "MAE": mae}

        row = i // 2
        col = i % 2

        ax = axes[row, col]
        scatter = ax.scatter(
            original,
            denoised,
            c=np.abs(original - denoised),
            cmap="viridis",
            s=20,
            alpha=0.7,
        )

        # Add identity line
        min_val = min(original.min(), denoised.min())
        max_val = max(original.max(), denoised.max())
        ax.plot([min_val, max_val], [min_val, max_val], "r--", alpha=0.7)

        ax.set_xlabel("Original Prediction")
        ax.set_ylabel("Denoised Prediction")
        ax.set_title(clock)

        # Add text annotation with correlation and MAE
        ax.text(
            0.05,
            0.95,
            f"Pearson: {pearson_corr:.4f}\nSpearman: {spearman_corr:.4f}\nMAE: {mae:.4f}",
            transform=ax.transAxes,
            verticalalignment="top",
            bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.8, "edgecolor": "gray"},
        )

        # Add colorbar
        plt.colorbar(scatter, ax=ax, label="Absolute Difference")

plt.tight_layout()
plt.show()

In [ ]:
# Define clocks of interest from pyaging
clocks = ["horvath2013", "altumage", "pchorvath2013", "skinandblood", "hannum"]

# Predict age using original beta values
adata_original = pya.pp.df_to_adata(df, imputer_strategy="mean", verbose=False)
pya.pred.predict_age(adata_original, clocks, verbose=False)

# Predict age using denoised beta values
adata_denoised = pya.pp.df_to_adata(X_denoised_df, imputer_strategy="mean", verbose=False)
pya.pred.predict_age(adata_denoised, clocks, verbose=False)

# Compare original and denoised predictions with ground truth age
comparison_results = {}

for clock in clocks:
    if clock in adata_original.obs.columns and clock in adata_denoised.obs.columns:
        original_pred = np.array(adata_original.obs[clock])
        denoised_pred = np.array(adata_denoised.obs[clock])
        true_age = np.array(df["age"])

        # Calculate metrics for original predictions
        original_mae = np.mean(np.abs(original_pred - true_age))
        original_rmse = np.sqrt(np.mean((original_pred - true_age) ** 2))
        original_r2 = np.corrcoef(true_age, original_pred)[0, 1] ** 2

        # Calculate metrics for denoised predictions
        denoised_mae = np.mean(np.abs(denoised_pred - true_age))
        denoised_rmse = np.sqrt(np.mean((denoised_pred - true_age) ** 2))
        denoised_r2 = np.corrcoef(true_age, denoised_pred)[0, 1] ** 2

        comparison_results[clock] = {
            "Original": {"MAE": original_mae, "RMSE": original_rmse, "R2": original_r2},
            "Denoised": {"MAE": denoised_mae, "RMSE": denoised_rmse, "R2": denoised_r2},
        }

# Create a DataFrame to display the results
comparison_df = pd.DataFrame.from_dict(
    {(i, j): comparison_results[i][j] for i in comparison_results for j in comparison_results[i]},
    orient="index",
)

# Visualize the comparison using matplotlib/seaborn
fig, axes = plt.subplots(3, 2, figsize=(10, 15))
fig.suptitle("Original vs Denoised Predictions Compared to True Age", fontsize=20)

for i, clock in enumerate(clocks):
    if clock in comparison_results:
        row = i // 2
        col = i % 2

        ax = axes[row, col]

        original_pred = np.array(adata_original.obs[clock])
        denoised_pred = np.array(adata_denoised.obs[clock])
        true_age = np.array(df["age"])

        # Plot original predictions
        ax.scatter(true_age, original_pred, color="blue", alpha=0.5, s=10, label="Original")

        # Plot denoised predictions
        ax.scatter(true_age, denoised_pred, color="red", alpha=0.5, s=10, label="Denoised")

        # Add identity line
        min_val = min(true_age.min(), original_pred.min(), denoised_pred.min())
        max_val = max(true_age.max(), original_pred.max(), denoised_pred.max())
        ax.plot([min_val, max_val], [min_val, max_val], "g--", alpha=0.7)

        ax.set_xlabel("True Age")
        ax.set_ylabel("Predicted Age")
        ax.set_title(clock)
        ax.legend()

        # Add text annotation with metrics
        ax.text(
            0.05,
            0.95,
            f"Original - MAE: {comparison_results[clock]['Original']['MAE']:.2f}, "
            f"R2: {comparison_results[clock]['Original']['R2']:.2f}\n"
            f"Denoised - MAE: {comparison_results[clock]['Denoised']['MAE']:.2f}, "
            f"R2: {comparison_results[clock]['Denoised']['R2']:.2f}",
            transform=ax.transAxes,
            verticalalignment="top",
            bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.8},
        )

plt.tight_layout()
plt.show()

## 10. Next steps

Here are some ideas for further exploration:

- Experiment with different input context lengths
- Use various pre-trained CpGPT models and compare how well they can denoise
- Predict age with other epigenetic clocks and compare the results